# Subsample datasets to 100 per year

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import random
import shutil 
import numpy as np
from unidecode import unidecode
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [2]:
# Set up directories

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Paloma/complete_human/"

references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references/"

# os.chdir(references)
# states_ref = pd.read_csv("states_ref.csv")

## Upload FASTAs

In [9]:
# Organize fastas

fastas = {}
for dirpath, dirs, files in os.walk(home):
    for file in files:
        file_name = os.path.join(dirpath, file) # .split("/")[-1]
        if "human_euro_pandemic_h1n1_ha_2009--2026.fasta" in file_name:
            fasta = df_from_fasta(file_name)
            fastas[file_name.split("/")[-1]] = fasta
    break # Do not go into subfolders

In [10]:
print(fastas)

{'human_euro_pandemic_h1n1_ha_2009--2026.fasta':                                              full_header  \
0      >EPI_ISL_100110|A/Austria/104/2011|H1N1|Austri...   
1      >EPI_ISL_230445|A/Khmelnitsky/667/2016|H1N1|Kh...   
2      >EPI_ISL_65700|A/Italy/215/2009|H1N1|Italy|200...   
3      >EPI_ISL_164087|A/Paris/5236/2009|H1N1|Paris|2...   
4      >EPI_ISL_263066|A/Sachsen/41/2017|H1N1|Sachsen...   
...                                                  ...   
60975  >EPI_ISL_19855954|A/Vologda/RII-MH233195S/2025...   
60976  >EPI_ISL_19855953|A/Vologda/RII-MH233194S/2025...   
60977  >EPI_ISL_19855952|A/Vologda/RII-MH233193S/2025...   
60978  >EPI_ISL_19855950|A/Vologda/RII-MH233177S/2025...   
60979  >EPI_ISL_19855949|A/Vologda/RII-MH233176S/2025...   

                                                sequence  
0      tttgcaaccgcaaatgcagacacattatgtataggttatcatgcga...  
1      ggaaaacaaaagcaacaaaaatgaaggcaatactagtagttctgct...  
2      actcgaacaaaggtgtaacggcagcatgtcctcatgctggagcaaa

## Subsample

In [11]:
# Create a dictionary of dictionaries of dataframes grouped by year

fastas_grouped = {} # Overall dictionary -- length is number of datasets needed

for key in fastas:
    years = {} # For each dataset, there are a number of years
    fasta = fastas[key]
    fasta["Year"] = fasta["full_header"].apply(lambda x: dateutil.parser.parse(x.split("|")[-2]).year) # Find year
    fasta["Isolate"] = fasta["full_header"].apply(lambda x: x.split("|")[1].lower()) # if "human" in x else x.split("/")[3] if "/" in x else "unknown") #  if len(x.split("/")) > 2 else "unknown")
    fasta = fasta.drop_duplicates(subset="Isolate", keep="first")
    for year, rows in fasta.groupby("Year"): # Separate dataframe into multiple dataframes by year
        years[year] = rows # For each year, there are a number of entries that have that year
    fastas_grouped[key] = years

print(fastas_grouped)

{'human_euro_pandemic_h1n1_ha_2009--2026.fasta': {2009:                                              full_header  \
2      >EPI_ISL_65700|A/Italy/215/2009|H1N1|Italy|200...   
3      >EPI_ISL_164087|A/Paris/5236/2009|H1N1|Paris|2...   
6      >EPI_ISL_67047|A/Catalonia/S2151/2009|H1N1|Spa...   
8      >EPI_ISL_66537|A/Moldova/G-170/2009|H1N1|Moldo...   
9      >EPI_ISL_65975|A/ENG/511/2009|H1N1|ENG|2009-06...   
...                                                  ...   
14105  >EPI_ISL_63741|A/ENG/255/2009|H1N1|ENG|2009-05...   
14106  >EPI_ISL_63740|A/ENG/251/2009|H1N1|ENG|2009-05...   
14107  >EPI_ISL_63739|A/ENG/XFL00176/2009|H1N1|ENG|20...   
14121  >EPI_ISL_31219|A/Netherlands/604/2009|H1N1|Net...   
14122  >EPI_ISL_31218|A/Netherlands/603/2009|H1N1|Net...   

                                                sequence  Year  \
2      actcgaacaaaggtgtaacggcagcatgtcctcatgctggagcaaa...  2009   
3      caacaaaaatgaaggcaatactagtagttctgctatatacatttgc...  2009   
6      cagtactagaaaagaatg

In [12]:
subsampled_dfs = {} # Overall subsampled dictionary -- length is number of datasets needed
for key in fastas_grouped:
    dataset = fastas_grouped[key] # Dataset
    df = pd.DataFrame() # Hold subsampled data
    for year_key in dataset: # Dictionary of years and their dataframes 
        year_df = dataset[year_key] # One year and its data
        # If there are more than 100 entries, subsample a random 100 
        subsampled = year_df[["full_header", "sequence"]].sample(n=100, random_state=2008) if len(year_df) > 100 else year_df[["full_header", "sequence"]]
        # print(subsampled)
        df = pd.concat([df, subsampled]) # Add subsampled data to dataframe
    subsampled_dfs[key] = df # Add dataframe to dictionary of datasets

print(subsampled_dfs)
        

{'human_euro_pandemic_h1n1_ha_2009--2026.fasta':                                              full_header  \
8923   >EPI_ISL_83411|A/Finland/634/2009|H1N1|Finland...   
13137  >EPI_ISL_62587|A/Norway/3779/2009|H1N1|Norway|...   
5153   >EPI_ISL_76228|A/Netherlands/1064b/2009|H1N1|N...   
933    >EPI_ISL_67570|A/Sachsen/92/2009|H1N1|Sachsen|...   
6343   >EPI_ISL_77516|A/Lisboa/75/2009|H1N1|Lisboa|20...   
...                                                  ...   
60145  >EPI_ISL_20346471|A/Netherlands/00148/2026|H1N...   
60267  >EPI_ISL_20347745|A/England/01898454/2026|H1N1...   
56347  >EPI_ISL_20339340|A/Netherlands/10092/2026|H1N...   
44777  >EPI_ISL_20349430|A/France/PDL-IPP00038/2026|H...   
49532  >EPI_ISL_20326893|A/Netherlands/10008/2026|H1N...   

                                                sequence  
8923   atgaaggcaatactagtagttctgctatatacatttgcaaccgcaa...  
13137  aaaacaaaagcaacaaaaatgaaggcaatactagtagttctgctat...  
5153   aaaagcaacaaaaatgaaggcaatactagtagttctgctatataca

## De-duplicate

In [13]:
for key in subsampled_dfs:
    fasta_df = subsampled_dfs[key]
    fasta_df["Accession"] = fasta_df["full_header"].apply(lambda x: x.split("|")[0].replace(">", ""))
    fasta_df["Host"] = fasta_df["full_header"].apply(lambda x: "human" if "human" in x else x.split("/")[1] if "/" in x else "unknown") #  if len(x.split("/")) > 1 else "unknown")
    fasta_df["Isolate"] = fasta_df["full_header"].apply(lambda x: x.split("/")[2] if "human" in x else x.split("/")[3] if "/" in x else "unknown") #  if len(x.split("/")) > 2 else "unknown")
    fasta_df["Subtype"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-4])
    fasta_df["Geo_Location"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-3])
    fasta_df["Collection_Date"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-2])
    fasta_df["Host_Type"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-1])
    
    print(fasta_df)

                                             full_header  \
8923   >EPI_ISL_83411|A/Finland/634/2009|H1N1|Finland...   
13137  >EPI_ISL_62587|A/Norway/3779/2009|H1N1|Norway|...   
5153   >EPI_ISL_76228|A/Netherlands/1064b/2009|H1N1|N...   
933    >EPI_ISL_67570|A/Sachsen/92/2009|H1N1|Sachsen|...   
6343   >EPI_ISL_77516|A/Lisboa/75/2009|H1N1|Lisboa|20...   
...                                                  ...   
60145  >EPI_ISL_20346471|A/Netherlands/00148/2026|H1N...   
60267  >EPI_ISL_20347745|A/England/01898454/2026|H1N1...   
56347  >EPI_ISL_20339340|A/Netherlands/10092/2026|H1N...   
44777  >EPI_ISL_20349430|A/France/PDL-IPP00038/2026|H...   
49532  >EPI_ISL_20326893|A/Netherlands/10008/2026|H1N...   

                                                sequence         Accession  \
8923   atgaaggcaatactagtagttctgctatatacatttgcaaccgcaa...     EPI_ISL_83411   
13137  aaaacaaaagcaacaaaaatgaaggcaatactagtagttctgctat...     EPI_ISL_62587   
5153   aaaagcaacaaaaatgaaggcaatactagtagttctgc

## Download FASTAs

In [14]:
# Prepare for download
for key in subsampled_dfs:
    file_name = "subsampled_" + key # Create file name
    fasta = subsampled_dfs[key]
    df_to_fasta(fasta, file_name, home)
